# Mid-update QC sweep — 2026 Q3 gas pipelines update

During-data-entry QC on the live "Pipelines (Gas/Oil/NGL) - main" sheet.
Every check lives in [`../qc/tracker_qc.py`](../qc/tracker_qc.py) — this
notebook is the interactive front end: it pulls the sheet, runs the checks,
and pages through the offending rows. The same module runs headless as
`python ../qc/tracker_qc.py --tracker ggit --cycle-start 2026-07-06`.

Run at least monthly, and after any bulk change.

**Read-only** — this notebook only reports; fix findings in the live sheet by
hand. Reads go through the sibling `gem-db-ops` repo (`gws` CLI, read-only
work profile), not the deleted `gem-analysis` service account.

In [ ]:
%pip install -q -e ../../gem-tracker-constants

In [ ]:
import sys

import pandas as pd

sys.path.insert(0, '../qc')
import tracker_qc

pd.set_option('display.max_rows', 250)
pd.set_option('display.max_colwidth', 60)

## Config

In [ ]:
# 'ggit' = the "Gas pipelines" tab of the live backend sheet. The tab, its key
# and its header-row offset all come from gem-db-ops/gem_sheets.py.
TRACKER = 'ggit'

# rows with LastUpdated on/after this date get the extra cycle checks
CYCLE_START = '2026-07-06'

# where to write the flagged-row worklist (gitignored — it's a data file)
WORKLIST_CSV = 'qc-findings.csv'

## Pull the live sheet

Read-only and authenticated, through `gem-db-ops/gem_sheets.py` — the single
source of truth for pulling GEM data. Needs the `gws` CLI authenticated
against `~/.config/gws-gem`; if auth has expired, re-run the login
interactively (it needs a browser).

Everything comes back as strings on purpose: QC has to see exactly what is in
each cell, including the values that fail to parse as numbers.

In [ ]:
df = tracker_qc.load_live(TRACKER)
print(f'{len(df)} rows x {len(df.columns)} columns')

## Run every check

`FAIL` = structurally wrong; it will corrupt a release or silently drop rows
from it (non-canonical vocabulary, a value sitting in the wrong column, a
derived column disagreeing with its source). `WARN` = a real data gap or
inconsistency to work through. `NOTE` = informational — legacy reference
sparseness, plausible-but-odd values, distributions worth an eyeball.

In [ ]:
findings = tracker_qc.run_all(df, cycle_start=CYCLE_START)
by_name = {f.name: f for f in findings}

tracker_qc.print_findings(findings, levels=('FAIL', 'WARN'))

In [ ]:
n_fail = tracker_qc.print_summary(findings)

## Work through the FAILs first

Each of these either breaks a release filter or means a value is in the wrong
column — cheap to fix, and expensive to ship.

In [ ]:
for f in sorted((f for f in findings if f.level == 'FAIL'), key=lambda f: -f.count):
    print(f'\n=== {f.name} — {f.count} rows ===')
    if f.detail:
        print(f.detail)
    display(f.rows(df))

## Rows touched this cycle

The checks scoped to rows stamped on/after `CYCLE_START`: missing required
fields and unsourced values in work done during this update. Sheet-wide
reference sparseness is legacy backlog and reports as `NOTE` — these are this
cycle's own gaps, and they are the ones worth chasing with researchers.

In [ ]:
for f in findings:
    if 'cycle' in f.name and f.count:
        print(f'\n=== [{f.level}] {f.name} — {f.count} rows ===')
        if f.detail:
            print(f.detail)
        display(f.rows(df).head(40))

## Look at any single check

`by_name` is keyed by the check names printed above; `.rows(df)` returns the
offending rows with the check's own columns attached.

In [ ]:
# every check that fired, so you can pick one
for f in findings:
    if f.count and f.level != 'OK':
        print(f'{f.level:5} {f.count:>6}  {f.name}')

In [ ]:
by_name['non-canonical Status'].rows(df)

## Export a worklist

One row per (check, ProjectID), so findings can be sorted by country or
researcher and handed out.

In [ ]:
worklist = tracker_qc.flagged_rows(df, findings)
worklist.to_csv(WORKLIST_CSV, index=False)
print(f'{len(worklist)} flagged rows -> {WORKLIST_CSV}')
worklist.groupby(['qc_tier', 'qc_level']).size().unstack(fill_value=0)

In [ ]:
# findings per country, to route them to the right researcher
(worklist.groupby(['CountriesOrAreas', 'qc_level']).size().unstack(fill_value=0)
 .assign(total=lambda d: d.sum(axis=1))
 .sort_values('total', ascending=False).head(30))